# BioDYM Material Flow Analysis - Scientific Notebook

A streamlined notebook for Material Flow Analysis using the BioDYM framework.

## Workflow
1. **Load Excel File** - Define input data
2. **Confirm Configuration** - Review loaded data and settings
3. **Run Calculation** - Execute MFA analysis
4. **Mass Balance Check** - Verify calculation accuracy
5. **Visualizations** - Display all available plots

---

## 1. Setup and Imports

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown

In [2]:
# Add BioDYM modules to path
src_path = os.path.join(os.getcwd(), 'src')
sys.path.insert(0, src_path)

In [3]:
# Add ODYM framework to path
biodym_mfa_tool_dir = os.getcwd()
odym_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "ODYM-master_20241127", "odym", "modules"
)
sys.path.insert(0, odym_path)

In [4]:
# Add bioDYM add-on to path
biodym_addon_path = os.path.join(
    biodym_mfa_tool_dir, "framework", "bioDYM_add-on", "modules"
)
sys.path.insert(0, biodym_addon_path)

In [5]:
# Import BioDYM modules
try:
    import config
    import data_loader
    import system_setup
    import utils
    from engine import solver
    import plotting
    print("✅ BioDYM modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

✅ BioDYM modules imported successfully


In [6]:
# Set up plotting
plt.style.use('default')
print("📊 Plotting environment ready")

📊 Plotting environment ready


## 2. Define Input File

**Change this variable to your Excel file:**

In [7]:
input_file = "data/01_input/250707_Template_CS1.xlsx"

In [8]:
print(f"📁 Input file: {input_file}")

📁 Input file: data/01_input/250707_Template_CS1.xlsx


## 3. Load and Validate Data

In [9]:
print("\n" + "="*60)
print("📊 LOADING AND VALIDATING DATA")
print("="*60)


📊 LOADING AND VALIDATING DATA


In [10]:
# Load Excel file
try:
    input_data = pd.read_excel(
        input_file,
        sheet_name=None,
        header=0,
        engine='openpyxl',
        na_values=['N.A.', 'NA', 'n/a']
    )
    print(f"✅ Excel file loaded: {len(input_data)} sheets")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    raise

c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

✅ Excel file loaded: 19 sheets


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

In [11]:
# Display sheet overview
print("\n📋 Sheet Overview:")
for sheet_name, df in input_data.items():
    print(f"   {sheet_name}: {df.shape[0]} rows × {df.shape[1]} columns")


📋 Sheet Overview:
   Version_CS0_07.07.: 0 rows × 0 columns
   0_ReadMe: 73 rows × 20 columns
   Table of Content: 26 rows × 2 columns
   0_Configuration: 25 rows × 3 columns
   1_1_Definition_Flows: 82 rows × 22 columns
   1_2_Data_Flows: 349 rows × 37 columns
   2_1_Definition_Processes: 57 rows × 38 columns
   2_3_Process_TCs: 63 rows × 21 columns
   2_4_Initial_Stock: 62 rows × 15 columns
   2_5_dynamic_tcs: 150 rows × 65 columns
   3_1_Definition_DSM: 50 rows × 18 columns
   3_2_Definition_FOMP: 46 rows × 13 columns
   4_1_Uncertainty_Parameters: 12 rows × 9 columns
   3. TC_Data: 0 rows × 0 columns
   PX - Template: 71 rows × 13 columns
   4. Calculation_factors>>>>: 0 rows × 0 columns
   4. Codelists>>>>: 0 rows × 1 columns
   4_1 Codelists: 49 rows × 12 columns
   5. Wastefiles >>>>: 0 rows × 1 columns


In [12]:
# Validate required sheets
required_sheets = [
    '1_1_Definition_Flows',
    '1_2_Data_Flows', 
    '2_1_Definition_Processes',
    '2_4_Initial_Stock',
    '2_5_dynamic_tcs'
]

In [13]:
missing_sheets = [sheet for sheet in required_sheets if sheet not in input_data.keys()]
if missing_sheets:
    print(f"\n⚠️ Missing required sheets: {missing_sheets}")
else:
    print("\n✅ All required sheets present")


✅ All required sheets present


## 4. Extract Configuration from Data

In [14]:
print("\n" + "="*60)
print("⚙️ EXTRACTING CONFIGURATION")
print("="*60)


⚙️ EXTRACTING CONFIGURATION


In [15]:
# Extract time range from flow data
flow_data = input_data['1_2_Data_Flows']
years = sorted(flow_data['Year_Flow'].unique())
start_year = int(min(years))
end_year = int(max(years))

In [16]:
print(f"📅 Time range: {start_year} - {end_year}")

📅 Time range: 2025 - 2050


In [17]:
# Extract elements from flow data
elements = ['material', 'WC', 'DM', 'CC']  # Default elements
print(f"🧪 Elements: {elements}")

🧪 Elements: ['material', 'WC', 'DM', 'CC']


In [18]:
# Check for Monte Carlo parameters
has_mc = '4_1_Uncertainty_Parameters' in input_data.keys()
print(f"🎲 Monte Carlo available: {'Yes' if has_mc else 'No'}")

🎲 Monte Carlo available: Yes


In [19]:
# Check for DSM parameters
has_dsm = '3_1_Definition_DSM' in input_data.keys()
print(f"📈 DSM available: {'Yes' if has_dsm else 'No'}")

📈 DSM available: Yes


In [20]:
# Check for FOMP parameters
has_fomp = '3_2_Definition_FOMP' in input_data.keys()
print(f"🌱 FOMP available: {'Yes' if has_fomp else 'No'}")

🌱 FOMP available: Yes


## 5. Confirm Configuration

In [21]:
print("\n" + "="*60)
print("✅ CONFIGURATION CONFIRMATION")
print("="*60)


✅ CONFIGURATION CONFIRMATION


In [22]:
config_summary = f"""
**Analysis Configuration:**
- Input File: {input_file}
- Time Range: {start_year} - {end_year}
- Elements: {', '.join(elements)}
- Monte Carlo: {'Enabled' if has_mc else 'Disabled'}
- DSM: {'Enabled' if has_dsm else 'Disabled'}
- FOMP: {'Enabled' if has_fomp else 'Disabled'}
"""

In [23]:
display(Markdown(config_summary))


**Analysis Configuration:**
- Input File: data/01_input/250707_Template_CS1.xlsx
- Time Range: 2025 - 2050
- Elements: material, WC, DM, CC
- Monte Carlo: Enabled
- DSM: Enabled
- FOMP: Enabled


## 6. Run MFA Calculation

In [24]:
print("\n" + "="*60)
print("🚀 RUNNING MFA CALCULATION")
print("="*60)


🚀 RUNNING MFA CALCULATION


In [25]:
# 1. Setup model scope
print("📋 Setting up model scope...")
try:
    model_classification, index_table = system_setup.define_model_scope(
        start_year, end_year, elements
    )
    print("✅ Model scope defined")
except Exception as e:
    print(f"❌ Error setting up model scope: {e}")
    raise

📋 Setting up model scope...
--> Model scope and classifications defined.
✅ Model scope defined


In [26]:
# 2. Initialize MFA system
print("🔧 Initializing MFA system...")
try:
    mfa_system_base = system_setup.initialize_mfa_system(
        model_classification, index_table
    )
    print("✅ MFA system initialized")
except Exception as e:
    print(f"❌ Error initializing MFA system: {e}")
    raise

🔧 Initializing MFA system...
--> MFA system object initialized.
✅ MFA system initialized


In [27]:
# 3. Load and define processes
print("📊 Loading processes and data...")
try:
    mfa_system_base, all_excel_data = system_setup.load_and_define_processes(
        mfa_system_base, input_file, data_loader
    )
    print("✅ Processes and data loaded")
except Exception as e:
    print(f"❌ Error loading processes: {e}")
    raise

📊 Loading processes and data...
--> Defining process and stock structures...


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Stock values initialized.
✅ Processes and data loaded


In [28]:
# 4. Load parameters
print("⚙️ Loading parameters...")
try:
    dsm_params = data_loader.load_dsm_parameters(all_excel_data)
    fomp_params = data_loader.load_fomp_parameters(all_excel_data)
    uncertainty_params = data_loader.load_uncertainty_definitions(all_excel_data)
    print("✅ Parameters loaded")
except Exception as e:
    print(f"❌ Error loading parameters: {e}")
    raise

⚙️ Loading parameters...
--> Loading DSM parameters from sheet '3_1_Definition_DSM'...
--> Successfully loaded configurations for 0 DSM process(es).
--> Loading FOMP parameters from sheet '3_2_Definition_FOMP'...
--> Successfully loaded configurations for 1 FOMP process(es).
--> Loading uncertainty definitions from sheet '4_1_Uncertainty_Parameters'...
--> Successfully loaded 4 uncertainty parameter definition(s).
✅ Parameters loaded


In [29]:
# 5. Define flows and parameters
print("🔗 Defining flows and parameters...")
try:
    mfa_system_configured, _ = system_setup.define_flows_and_parameters(
        mfa_system_base, all_excel_data
    )
    print(f"✅ System configured: {len(mfa_system_configured.ProcessList)} processes, "
          f"{len(mfa_system_configured.FlowDict)} flows, {len(mfa_system_configured.StockDict)} stocks")
except Exception as e:
    print(f"❌ Error defining flows and parameters: {e}")
    raise

🔗 Defining flows and parameters...
--> Defining flows, parameters, and setting all initial values...
--> All flows initialized to zero.
--> Populated data for primary input flows.
✅ System configured: 11 processes, 18 flows, 8 stocks


In [30]:
# 6. Run calculation
print("🧮 Running calculation...")
try:
    mfa_system_with_results, dsm_details = solver.run_mfa_calculation(
        mfa_system_configured, dsm_params, fomp_params, config
    )
    print("✅ Calculation completed successfully!")
except Exception as e:
    print(f"❌ Calculation error: {e}")
    import traceback
    traceback.print_exc()
    raise

🧮 Running calculation...
--> Calculating final stock balances for ALL processes...
--> Stock balance calculation finished.
✅ Calculation completed successfully!


## 7. Mass Balance Check

In [31]:
print("\n" + "="*60)
print("⚖️ MASS BALANCE VERIFICATION")
print("="*60)


⚖️ MASS BALANCE VERIFICATION


In [32]:
# Calculate mass balance errors
mass_balance_errors = []
for process in mfa_system_with_results.ProcessList:
    if hasattr(process, 'MassBalance') and process.MassBalance is not None:
        for year_idx, year in enumerate(range(start_year, end_year + 1)):
            for element_idx, element in enumerate(elements):
                error = process.MassBalance[year_idx, element_idx]
                if abs(error) > 1e-6:  # Significant error threshold
                    mass_balance_errors.append({
                        'Process': process.Name,
                        'Year': year,
                        'Element': element,
                        'Error': error
                    })

In [33]:
if mass_balance_errors:
    print("⚠️ Mass balance errors detected:")
    error_df = pd.DataFrame(mass_balance_errors)
    display(error_df)
else:
    print("✅ All mass balances within acceptable limits")

✅ All mass balances within acceptable limits


## 8. Results Overview

In [34]:
print("\n" + "="*60)
print("📈 RESULTS OVERVIEW")
print("="*60)


📈 RESULTS OVERVIEW


In [35]:
# Display final stock values
print("\n📊 Final Stock Values (Year {end_year}):")
final_stocks = []
for stock_name, stock in mfa_system_with_results.StockDict.items():
    if stock_name.startswith('S_'):  # Absolute stocks only
        final_value = stock.Values[-1, 0]  # Material dimension, final year
        final_stocks.append({
            'Stock': stock_name,
            'Final Value (Mg)': final_value
        })


📊 Final Stock Values (Year {end_year}):


In [36]:
if final_stocks:
    stocks_df = pd.DataFrame(final_stocks)
    display(stocks_df)

,Stock,Final Value (Mg)
0,S_0,-129.623475
1,S_1,-795.000000
2,S_6,0.000000
3,S_10,924.623475


In [37]:
# Display flow summary
print("\n🔄 Flow Summary:")
flow_summary = []
for flow_id, flow in mfa_system_with_results.FlowDict.items():
    avg_flow = np.mean(flow.Values[:, 0])  # Average material flow
    flow_summary.append({
        'Flow ID': flow_id,
        'From': flow.P_Start,
        'To': flow.P_End,
        'Avg Flow (Mg/year)': avg_flow
    })


🔄 Flow Summary:


In [38]:
if flow_summary:
    flows_df = pd.DataFrame(flow_summary)
    display(flows_df.head(10))  # Show first 10 flows

,Flow ID,From,To,Avg Flow (Mg/year)
0,F_00_02,0,2,217.307692
1,F_01_02,1,2,217.307692
2,F_02_03,2,3,434.615385
3,F_03_04,3,4,217.307692
4,F_03_05,3,5,217.307692
5,F_04_00,4,0,108.653846
6,F_04_01,4,1,108.653846
7,F_05_06,5,6,86.923077
8,F_06_07,6,7,86.923077
9,F_07_00,7,0,43.461538


## 9. Visualizations

In [39]:
print("\n" + "="*60)
print("📊 VISUALIZATIONS")
print("="*60)


📊 VISUALIZATIONS


In [40]:
# 1. Mass Balance Error Plot
print("\n1️⃣ Mass Balance Error Plot:")
try:
    plotting.plot_mass_balance_error(mfa_system_with_results)
    plt.show()
except Exception as e:
    print(f"⚠️ Could not create mass balance plot: {e}")


1️⃣ Mass Balance Error Plot:


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'marker': {'color': [#7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f,
                                   #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f, #7f7f7f,
                                   #7f7f7f]},
              'type': 'bar',
              'uid': '2f20fc2f-f866-4b3e-90f7-16de7f035b65',
              'x': [Atmosphere, Environment, Cultivation, Harvest, Grain
                    Processing & Consumption, Straw d&C, Utilization in
                    construction, Incineration, Incorporation, Animal bedding,
                    Lithosphere],
              'y': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}],
    'layout': {'height': 500,
               'shapes': [{'line': {'color': 'black', 'width': 2},
                           'type': 'line',
                           'x0': -0.5,
                           'x1': 10.5,
                           'y0': 0,
                           'y1': 0}],
               'template': '...',
               'title': {'t

In [41]:
# 2. Stock Dynamics
print("\n2️⃣ Stock Dynamics Over Time:")
try:
    fig = go.Figure()
    
    for stock_name, stock in mfa_system_with_results.StockDict.items():
        if stock_name.startswith('S_'):  # Only absolute stocks
            years = list(range(start_year, end_year + 1))
            fig.add_trace(go.Scatter(
                x=years,
                y=stock.Values[:, 0],  # Material dimension
                mode='lines+markers',
                name=stock_name,
                line=dict(width=2)
            ))
    
    fig.update_layout(
        title='Stock Dynamics Over Time',
        xaxis_title='Year',
        yaxis_title='Stock Value (Mg)',
        showlegend=True,
        height=500
    )
    fig.show()
except Exception as e:
    print(f"⚠️ Could not create stock dynamics plot: {e}")


2️⃣ Stock Dynamics Over Time:


In [42]:
# 3. Flow Diagram (Simplified)
print("\n3️⃣ Material Flow Diagram:")
try:
    fig = go.Figure()
    
    # Create a simple flow representation
    for flow_id, flow in mfa_system_with_results.FlowDict.items():
        avg_flow = np.mean(flow.Values[:, 0])
        if avg_flow > 0:  # Only show significant flows
            fig.add_trace(go.Scatter(
                x=[flow.P_Start, flow.P_End],
                y=[0, 0],
                mode='lines+markers',
                name=f"{flow_id} ({avg_flow:.1f} Mg/year)",
                line=dict(width=avg_flow/10),  # Line width proportional to flow
                showlegend=True
            ))
    
    fig.update_layout(
        title='Material Flow Diagram',
        xaxis_title='Process ID',
        yaxis_title='Flow Value',
        showlegend=True,
        height=400
    )
    fig.show()
except Exception as e:
    print(f"⚠️ Could not create flow diagram: {e}")


3️⃣ Material Flow Diagram:


In [43]:
# 4. Process Efficiency (if DSM available)
if has_dsm and dsm_details:
    print("\n4️⃣ Process Efficiency (DSM):")
    try:
        # This would show DSM-specific visualizations
        print("DSM results available - additional plots could be added here")
    except Exception as e:
        print(f"⚠️ Could not create DSM plots: {e}")

## 10. Export Results

In [44]:
print("\n" + "="*60)
print("💾 EXPORTING RESULTS")
print("="*60)


💾 EXPORTING RESULTS


In [45]:
# Export to Excel
output_file = "data/02_output/results_scientific.xlsx"
try:
    utils.export_results_to_excel(mfa_system_with_results, output_file)
    print(f"✅ Results exported to: {output_file}")
except Exception as e:
    print(f"⚠️ Export error: {e}")

--> Exporting results to 'data/02_output/results_scientific.xlsx'...
✅ Results exported to: data/02_output/results_scientific.xlsx


In [46]:
# Export configuration summary
config_file = output_file.replace('.xlsx', '_config.xlsx')
try:
    config_summary = pd.DataFrame([{
        'Input File': input_file,
        'Start Year': start_year,
        'End Year': end_year,
        'Elements': ', '.join(elements),
        'Monte Carlo': has_mc,
        'DSM': has_dsm,
        'FOMP': has_fomp
    }])
    config_summary.to_excel(config_file, index=False)
    print(f"✅ Configuration exported to: {config_file}")
except Exception as e:
    print(f"⚠️ Config export error: {e}")

✅ Configuration exported to: data/02_output/results_scientific_config.xlsx


## 11. Summary

In [47]:
print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE")
print("="*60)


🎉 ANALYSIS COMPLETE


In [48]:
summary = f"""
**Analysis Summary:**
- ✅ Input file processed successfully
- ✅ Configuration extracted automatically
- ✅ MFA calculation completed
- ✅ Mass balance verified
- ✅ Visualizations generated
- ✅ Results exported

**Key Results:**
- Time period: {start_year} - {end_year}
- Processes analyzed: {len(mfa_system_with_results.ProcessList)}
- Flows tracked: {len(mfa_system_with_results.FlowDict)}
- Stocks modeled: {len(mfa_system_with_results.StockDict)}
- Mass balance errors: {len(mass_balance_errors)}

**Files Generated:**
- Main results: {output_file}
- Configuration: {config_file}
"""

In [49]:
display(Markdown(summary))


**Analysis Summary:**
- ✅ Input file processed successfully
- ✅ Configuration extracted automatically
- ✅ MFA calculation completed
- ✅ Mass balance verified
- ✅ Visualizations generated
- ✅ Results exported

**Key Results:**
- Time period: 2025 - 2050
- Processes analyzed: 11
- Flows tracked: 18
- Stocks modeled: 8
- Mass balance errors: 0

**Files Generated:**
- Main results: data/02_output/results_scientific.xlsx
- Configuration: data/02_output/results_scientific_config.xlsx


In [50]:
print("\n📊 Analysis completed successfully!") 


📊 Analysis completed successfully!
